# Homework 3

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Dataset

In [2]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
!wget $data -O data_homework_3.csv

--2026-10-12 15:50:24--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolvendo raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Conectando-se a raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... conectado.
A requisição HTTP foi enviada, aguardando resposta... 200 OK
Tamanho: 278746 (272K) [text/plain]
Salvando em: ‘data_homework_3.csv’

data_homework_3.csv 100%[===================>] 272,21K  --.-KB/s    em 0,07s   

2026-10-12 15:50:24 (3,76 MB/s) - ‘data_homework_3.csv’ salvo [278746/278746]



In [3]:
df = pd.read_csv('data_homework_3.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [4]:
df.dtypes

lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

# Data preparation
* Check if the missing values are presented in the features
* if there are missing values:
  - For categorical features, replace then with 'NA'
  - For numerical features, replace with 0.0

In [5]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [6]:
base = ['annual_income', 'lead_score']
df[base] = df[base].fillna(0)
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income                 0
number_of_courses_viewed      0
interaction_count             0
lead_score                    0
converted                     0
dtype: int64

In [7]:
base = ['lead_source', 'industry', 'employment_status', 'location']
df[base] = df[base].fillna('NA')
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [8]:
print(type(df))

<class 'pandas.core.frame.DataFrame'>


# Question 1

## What is the most frequent observation (mode) for the column industry?

In [9]:
print(df.industry.value_counts())
print()
print('mode = ', df.industry.mode()[0])

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

mode =  technology


### Question 2

Create the [correlation matrix](https://www.google.com/search?q=correlation+matrix) for the numerical features of your dataset.
In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

- `interaction_count` and `lead_score`
- `number_of_courses_viewed` and `lead_score`
- `number_of_courses_viewed` and `interaction_count`
- `annual_income` and `interaction_count`

Only consider the pairs above when answering this question.



In [10]:
df.columns

Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted'],
      dtype='object')

In [11]:
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted']

df[numerical].dtypes

df[numerical].corr(method='pearson', min_periods=1, numeric_only=True)

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


### Question 2 answer: The biggest correlation is lead_score versus interaction_count 

### Split the data

- Split the data with these exact calls:

```python
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)
```

- Make sure that the target value `converted` is not in your dataframe.

In [12]:
from sklearn.model_selection import train_test_split

In [13]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [14]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [15]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [16]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

### Question 3

- Calculate the mutual information score between `converted` and other categorical variables in the dataset. Use the training set only.
- Round the scores to 2 decimals using `round(score, 2)`.

Which of these variables has the biggest mutual information score?

- `industry`
- `location`
- `lead_source`
- `employment_status`


In [17]:
from sklearn.metrics import mutual_info_score

In [18]:
df.dtypes
df.columns

Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted'],
      dtype='object')

In [19]:
categorical = ['lead_source', 'industry', 'employment_status', 'location']

In [20]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

mi = df_train[categorical].apply(mutual_info_converted_score)
print (round(mi.sort_values(ascending = False), 2))

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64


### `lead_source`has the highest mutual information score with respect to `converted`.

### Question 4

- Now let's train a logistic regression.
- Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
- Fit the model on the training dataset.
  - To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
  - `model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)`
- Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?

- 0.55
- 0.65
- 0.75
- 0.85

In [21]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [22]:
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score']

categorical = ['lead_source', 'industry', 'employment_status', 'location']

train_dicts = df_train[categorical + numerical].to_dict(orient='records')

dv = DictVectorizer(sparse=False)

X_train = dv.fit_transform(train_dicts)

X_train[0]

array([6.598e+04, 0.000e+00, 1.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 0.000e+00, 1.000e+00, 0.000e+00, 0.000e+00, 0.000e+00,
       0.000e+00, 3.000e+00, 3.700e-01, 0.000e+00, 0.000e+00, 0.000e+00,
       1.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 0.000e+00, 1.000e+00,
       0.000e+00, 0.000e+00, 0.000e+00, 2.000e+00])

In [23]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

print(model.intercept_[0].round(3), '\n')
print(model.coef_[0].round(3))

-0.011 

[-0.    -0.001  0.007 -0.003 -0.008 -0.006 -0.    -0.003 -0.001 -0.002
 -0.002 -0.002 -0.001  0.137  0.011  0.    -0.009  0.007 -0.014  0.009
 -0.004  0.    -0.002 -0.002 -0.001 -0.003 -0.004  0.054]


In [24]:
y_pred = model.predict_proba(X_train)[:, 1]
y_pred

array([0.54986385, 0.41492578, 0.64935551, ..., 0.62831068, 0.44168571,
       0.48544926], shape=(3000,))

In [25]:
val_dicts = df_val[categorical + numerical].to_dict(orient='records')

X_val = dv.transform(val_dicts)

In [26]:
y_pred = model.predict_proba(X_val)[:, 1]

converted_decision = y_pred >= 0.5

converted_decision[:5]

array([ True, False,  True,  True,  True])

In [27]:
print((y_val == converted_decision)[:10], '\n')
print(round((y_val == converted_decision).mean(),3), '\n')

[ True False False  True False  True  True False  True  True] 

0.645 



### Answer of question 4: The accuracy is 0.65.

### Question 5

- Let's find the least useful feature using the _feature elimination_ technique.
- Train a model using the same features and parameters as in Q4 (without rounding).
- Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
- For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

Which of following feature has the smallest difference?

- `'lead_source'`
- `'number_of_courses_viewed'`
- `'interaction_count'`

> **Note**: The difference doesn't have to be positive.

In [31]:
precision_diff = []

base = ['lead_source', 'industry', 'employment_status', 'location', 'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score']

# start the model:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

# training:
train_dicts = df_train[base].to_dict(orient='records')
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
model.fit(X_train, y_train)
y_pred = model.predict_proba(X_train)[:, 1]

# validation
val_dicts = df_val[base].to_dict(orient='records')
X_val = dv.transform(val_dicts)

# precision:
y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = y_pred >= 0.5
precision_all = (y_val == converted_decision).mean()

# loop to exclude each feature:
for feature in base:
    features = [f for f in base if f != feature]
    # training:
    train_dicts = df_train[features].to_dict(orient='records')
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(train_dicts)
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_train)[:, 1]

    # validation
    val_dicts = df_val[features].to_dict(orient='records')
    X_val = dv.transform(val_dicts)

    # precision:
    y_pred = model.predict_proba(X_val)[:, 1]
    converted_decision = y_pred >= 0.5
    precision_diff.append((feature, (precision_all - (y_val == converted_decision).mean())))

# sort the precision_diff:
precision_diff = sorted(precision_diff, key=lambda x:x[1], reverse=False)

# print the results:
for name, value in precision_diff:
    if name in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
        print(f'{value:.5f}: without {name}')

0.00200: without number_of_courses_viewed
0.00300: without lead_source
0.04400: without interaction_count


### Answer of question 5: `number_of_courses_viewed` has the smallest difference.

### Question 6

- Now let's train a regularized logistic regression.
- Let's try the following values of the parameter `C`: `[0.000001, 0.00001, 0.0001, 0.001]`.
- Train models using all the features as in Q4.
- Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these `C` leads to the best accuracy on the validation set?

- 0.000001
- 0.00001
- 0.0001
- 0.001

> **Note**: If there are multiple options, select the smallest `C`.

In [33]:
base = ['lead_source', 'industry', 'employment_status', 'location', 'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score']

def LogisticRegression_function(C, df_train, y_train, df_val, y_val, base):  

    # start the model:
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)

    # training:
    train_dicts = df_train[base].to_dict(orient='records')
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(train_dicts)
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_train)[:, 1]

    # validation
    val_dicts = df_val[base].to_dict(orient='records')
    X_val = dv.transform(val_dicts)

    # precision:
    y_pred = model.predict_proba(X_val)[:, 1]
    converted_decision = y_pred >= 0.5
    precision_all = (y_val == converted_decision).mean()

    return precision_all


In [34]:
accuracy = []

for C in [0.000001, 0.00001, 0.0001, 0.001]:
    
    precision = LogisticRegression_function(C, df_train, y_train, df_val, y_val, base)
    accuracy.append((C, precision))

# sort the accuracy:
accuracy = sorted(accuracy, key=lambda x:x[1], reverse=True)

# print the results:
for C, value in accuracy:
    print(f'C = {C:.6f}: accuracy = {value:.3f}')

C = 0.001000: accuracy = 0.645
C = 0.000100: accuracy = 0.613
C = 0.000001: accuracy = 0.598
C = 0.000010: accuracy = 0.598


### Answer of question 6: the best accuracy is`C=0.001`.